# Prompt Analytics: Measuring Prompt Performance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/12-meta-prompting/102_prompt_analytics.ipynb)

**Category**: 12 - Meta-Prompting | **Technique #102**

---

Prompt Analytics provides systematic measurement and analysis of prompt performance through metrics like accuracy, latency, cost, and user satisfaction to drive continuous improvement.

## Description

Prompt Analytics enables:
- Performance measurement and tracking
- Cost optimization insights
- Quality trend analysis
- Comparative analysis between versions
- Data-driven optimization decisions

**When to use:**
- Production prompt monitoring
- Cost optimization initiatives
- A/B testing analysis
- Prompt optimization projects
- ROI reporting on AI investments

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                   PROMPT ANALYTICS PIPELINE                 │
└─────────────────────────────────────────────────────────────┘

  Prompt Execution
         │
         ▼
  ┌─────────────────────────────────────────────────────┐
  │              METRICS COLLECTION                     │
  │  ┌──────────┐ ┌──────────┐ ┌──────────┐            │
  │  │ Quality  │ │  Cost    │ │  Speed   │            │
  │  │  Metrics │ │  Metrics │ │  Metrics │            │
  │  └──────────┘ └──────────┘ └──────────┘            │
  └─────────────────────────────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────────────────────────────┐
  │              ANALYSIS & VISUALIZATION               │
  │  - Trends over time                                 │
  │  - Comparative analysis                             │
  │  - Anomaly detection                                │
  │  - Correlation analysis                             │
  └─────────────────────────────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────────────────────────────┐
  │              ACTIONABLE INSIGHTS                    │
  │  - Optimization recommendations                     │
  │  - Cost reduction opportunities                     │
  │  - Quality improvement areas                        │
  └─────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
# Install required packages
!pip install openai matplotlib -q

import os
import json
import time
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from collections import defaultdict
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## Basic Example: Simple Analytics Collector

In [ ]:
@dataclass
class PromptExecution:
    """Record of a single prompt execution."""
    prompt_id: str
    timestamp: datetime
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model: str
    success: bool
    error_message: Optional[str] = None
    custom_metrics: Dict[str, Any] = field(default_factory=dict)

class PromptAnalytics:
    """Simple analytics collector."""
    
    # Pricing per 1K tokens (approximate)
    PRICING = {
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015}
    }
    
    def __init__(self):
        self.executions: List[PromptExecution] = []
    
    def record(
        self,
        prompt_id: str,
        input_tokens: int,
        output_tokens: int,
        latency_ms: float,
        model: str,
        success: bool = True,
        error_message: str = None,
        custom_metrics: Dict = None
    ):
        """Record an execution."""
        execution = PromptExecution(
            prompt_id=prompt_id,
            timestamp=datetime.now(),
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            latency_ms=latency_ms,
            model=model,
            success=success,
            error_message=error_message,
            custom_metrics=custom_metrics or {}
        )
        self.executions.append(execution)
    
    def calculate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost for a request."""
        pricing = self.PRICING.get(model, self.PRICING["gpt-4o"])
        input_cost = (input_tokens / 1000) * pricing["input"]
        output_cost = (output_tokens / 1000) * pricing["output"]
        return input_cost + output_cost
    
    def get_summary(self, prompt_id: Optional[str] = None) -> Dict[str, Any]:
        """Get analytics summary."""
        executions = [e for e in self.executions if not prompt_id or e.prompt_id == prompt_id]
        
        if not executions:
            return {"error": "No data available"}
        
        total_cost = sum(
            self.calculate_cost(e.model, e.input_tokens, e.output_tokens)
            for e in executions
        )
        
        return {
            "total_executions": len(executions),
            "successful": sum(1 for e in executions if e.success),
            "failed": sum(1 for e in executions if not e.success),
            "avg_latency_ms": sum(e.latency_ms for e in executions) / len(executions),
            "total_tokens": sum(e.input_tokens + e.output_tokens for e in executions),
            "total_cost_usd": round(total_cost, 4),
            "avg_cost_per_request": round(total_cost / len(executions), 6),
            "models_used": list(set(e.model for e in executions))
        }

# Create analytics instance
analytics = PromptAnalytics()

# Simulate some executions
for i in range(5):
    start = time.time()
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Say hello {i}"}],
        max_tokens=50
    )
    
    latency = (time.time() - start) * 1000
    
    analytics.record(
        prompt_id="greeting_prompt",
        input_tokens=response.usage.prompt_tokens,
        output_tokens=response.usage.completion_tokens,
        latency_ms=latency,
        model="gpt-4o-mini",
        success=True
    )

# Get summary
summary = analytics.get_summary("greeting_prompt")
print("=== ANALYTICS SUMMARY ===\n")
for key, value in summary.items():
    print(f"{key}: {value}")

## Real-World Example: Comprehensive Analytics Dashboard

In [ ]:
class ComprehensiveAnalytics:
    """Production-grade analytics system."""
    
    def __init__(self):
        self.executions: List[PromptExecution] = []
        self.prompt_metadata: Dict[str, Dict] = {}
    
    def register_prompt(self, prompt_id: str, metadata: Dict):
        """Register a prompt with metadata."""
        self.prompt_metadata[prompt_id] = metadata
    
    def record(self, execution: PromptExecution):
        """Record an execution."""
        self.executions.append(execution)
    
    def get_prompt_comparison(self, prompt_ids: List[str]) -> Dict:
        """Compare multiple prompts."""
        comparison = {}
        
        for pid in prompt_ids:
            execs = [e for e in self.executions if e.prompt_id == pid]
            if execs:
                total_cost = sum(
                    (e.input_tokens / 1000) * 0.005 + (e.output_tokens / 1000) * 0.015
                    for e in execs
                )
                comparison[pid] = {
                    "executions": len(execs),
                    "avg_latency_ms": round(sum(e.latency_ms for e in execs) / len(execs), 2),
                    "success_rate": sum(1 for e in execs if e.success) / len(execs),
                    "total_cost": round(total_cost, 4),
                    "avg_tokens_per_request": sum(e.input_tokens + e.output_tokens for e in execs) / len(execs)
                }
        
        return comparison
    
    def get_time_series(self, prompt_id: str, metric: str = "latency_ms") -> List[Dict]:
        """Get time series data for a metric."""
        execs = [e for e in self.executions if e.prompt_id == prompt_id]
        return [
            {
                "timestamp": e.timestamp.isoformat(),
                "value": getattr(e, metric, e.custom_metrics.get(metric, 0))
            }
            for e in sorted(execs, key=lambda x: x.timestamp)
        ]
    
    def detect_anomalies(self, prompt_id: str, metric: str = "latency_ms", threshold: float = 2.0) -> List[Dict]:
        """Detect anomalous executions."""
        execs = [e for e in self.executions if e.prompt_id == prompt_id]
        if len(execs) < 5:
            return []
        
        values = [getattr(e, metric, 0) for e in execs]
        mean = sum(values) / len(values)
        std = (sum((v - mean) ** 2 for v in values) / len(values)) ** 0.5
        
        anomalies = []
        for e in execs:
            val = getattr(e, metric, 0)
            if abs(val - mean) > threshold * std:
                anomalies.append({
                    "timestamp": e.timestamp.isoformat(),
                    "value": val,
                    "expected_range": [mean - threshold * std, mean + threshold * std]
                })
        
        return anomalies
    
    def generate_dashboard(self) -> str:
        """Generate dashboard summary."""
        total_execs = len(self.executions)
        total_cost = sum(
            (e.input_tokens / 1000) * 0.005 + (e.output_tokens / 1000) * 0.015
            for e in self.executions
        )
        
        dashboard = f"""
╔═══════════════════════════════════════════════════════════════╗
║                    ANALYTICS DASHBOARD                        ║
╠═══════════════════════════════════════════════════════════════╣
║  Total Executions:    {total_execs:>10}                               ║
║  Success Rate:        {sum(1 for e in self.executions if e.success)/total_execs*100 if total_execs else 0:>9.1f}%                              ║
║  Total Cost:          ${total_cost:>9.4f}                             ║
║  Avg Latency:         {sum(e.latency_ms for e in self.executions)/total_execs if total_execs else 0:>9.0f}ms                             ║
║  Active Prompts:      {len(self.prompt_metadata):>10}                               ║
╚═══════════════════════════════════════════════════════════════╝
"""
        return dashboard

# Create comprehensive analytics
comp_analytics = ComprehensiveAnalytics()

# Register prompts
comp_analytics.register_prompt("summarizer", {"purpose": "Text summarization", "owner": "team-a"})
comp_analytics.register_prompt("classifier", {"purpose": "Content classification", "owner": "team-b"})

# Simulate executions for different prompts
import random

for i in range(20):
    prompt_id = random.choice(["summarizer", "classifier"])
    
    # Simulate varying performance
    base_latency = 500 if prompt_id == "summarizer" else 200
    latency = base_latency + random.gauss(0, 100)
    
    exec_record = PromptExecution(
        prompt_id=prompt_id,
        timestamp=datetime.now() - timedelta(minutes=i*5),
        input_tokens=random.randint(100, 500),
        output_tokens=random.randint(50, 200),
        latency_ms=max(50, latency),
        model="gpt-4o-mini",
        success=random.random() > 0.05,  # 95% success rate
        custom_metrics={"user_rating": random.randint(3, 5)}
    )
    comp_analytics.record(exec_record)

# Display dashboard
print(comp_analytics.generate_dashboard())

# Compare prompts
print("\n=== PROMPT COMPARISON ===")
comparison = comp_analytics.get_prompt_comparison(["summarizer", "classifier"])
for pid, metrics in comparison.items():
    print(f"\n{pid}:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

# Detect anomalies
print("\n=== ANOMALY DETECTION ===")
anomalies = comp_analytics.detect_anomalies("summarizer", "latency_ms")
print(f"Found {len(anomalies)} anomalies for summarizer")
for a in anomalies[:3]:
    print(f"  {a['timestamp']}: {a['value']:.0f}ms")

## Failure Case: Analytics Pitfalls

In [ ]:
print("=== COMMON ANALYTICS PITFALLS ===\n")

print("1. VANITY METRICS")
print("   Bad:  Tracking total requests only")
print("   Good:  Track success rate, cost per success")
print("   Why:   Volume doesn't indicate quality\n")

print("2. MISSING CONTEXT")
print("   Bad:  Latency increased by 20%")
print("   Good:  Latency increased 20% after model upgrade")
print("   Why:   Context explains the 'why'\n")

print("3. AGGREGATION BLINDNESS")
print("   Bad:  Average latency is 500ms")
print("   Good:  P50: 300ms, P95: 1200ms, P99: 2000ms")
print("   Why:   Averages hide outliers\n")

print("4. NO BASELINE COMPARISON")
print("   Bad:  Current cost is $100/day")
print("   Good:  Cost reduced 30% from $143/day baseline")
print("   Why:   Need reference point for evaluation\n")

print("5. IGNORING SEASONALITY")
print("   Bad:  Tuesday traffic was 50% higher")
print("   Good:  Tuesday traffic 10% above weekly average")
print("   Why:   Day-of-week effects are normal\n")

print("="*60)
print("BEST PRACTICES:")
print("• Focus on actionable metrics")
print("• Include percentiles, not just averages")
print("• Track cost per successful outcome")
print("• Compare against baselines")
print("• Correlate with business metrics")

## Benchmark: Key Metrics to Track

| Category | Metric | Target | Alert Threshold |
|----------|--------|--------|-----------------|
| Quality | Success Rate | >95% | <90% |
| Quality | User Satisfaction | >4.0/5 | <3.5/5 |
| Cost | Cost/Request | Minimize | >2x baseline |
| Speed | P50 Latency | <500ms | >1000ms |
| Speed | P99 Latency | <2000ms | >5000ms |
| Efficiency | Tokens/Request | Optimize | >2x baseline |

**Dashboard Refresh**: Real-time for critical metrics, hourly for trends.

## Interactive Playground

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE PLAYGROUND                     ║
# ╚═══════════════════════════════════════════════════════════════╝

# Create your own analytics tracking
my_analytics = ComprehensiveAnalytics()

# Register your prompts
# my_analytics.register_prompt("my_prompt", {"purpose": "description"})

# Record executions (integrate with your code)
# my_analytics.record(PromptExecution(...))

# Generate reports
# print(my_analytics.generate_dashboard())
# comparison = my_analytics.get_prompt_comparison(["prompt1", "prompt2"])

## Tips & Tricks

### Cost Optimization Strategies

| Strategy | Potential Savings | Implementation |
|----------|------------------|----------------|
| Model downgrading | 50-90% | Use cheaper models for simple tasks |
| Caching | 20-40% | Cache frequent queries |
| Prompt compression | 10-30% | Remove unnecessary tokens |
| Response streaming | 15-25% | Show partial results early |
| Batch processing | 30-50% | Group similar requests |

### Alerting Rules

```python
# Example alert conditions
alerts = {
    "error_rate": lambda metrics: metrics["error_rate"] > 0.05,
    "latency_p99": lambda metrics: metrics["latency_p99"] > 5000,
    "cost_spike": lambda metrics: metrics["cost"] > 2 * metrics["baseline_cost"],
    "success_rate_drop": lambda metrics: metrics["success_rate"] < 0.90
}
```

## References

1. [OpenAI Usage Dashboard](https://platform.openai.com/usage)
2. [LangSmith Monitoring](https://docs.smith.langchain.com/monitoring)
3. [Weights & Biases LLM Monitoring](https://docs.wandb.ai/guides/prompts)
4. [Helicone LLM Observability](https://docs.helicone.ai/)

---

**Previous**: [101_prompt_testing_framework.ipynb](101_prompt_testing_framework.ipynb)

---

## Category 12 Complete!

You've learned all 8 meta-prompting techniques. Continue to the next category or review previous notebooks.